In [0]:
# Databricks Notebook
# 6009 Assignment 1 - Step 1 Data Cleaning (Yellow Taxi Data Denoising)

# Environment Information
print("Spark Version:", spark.version)
print("Platform: Databricks")
print("="*50)

# Import packages
from pyspark.sql import functions as F

# ====================== Yellow Taxi Merged Dataset Path ======================
file_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/combined data/yellow_final_merged.parquet"

# 1. Read raw data
df = spark.read.parquet(file_path)
print("Raw data row count:", df.count())

# 2. Data denoising
df_clean = df.distinct() \
             .filter(F.col("tpep_pickup_datetime").isNotNull()) \
             .filter(F.col("tpep_dropoff_datetime").isNotNull()) \
             .filter(F.col("trip_distance") > 0) \
             .filter(F.col("fare_amount") > 0) \
             .filter(F.col("fare_amount") <= 1000) \
             .filter(F.col("passenger_count").between(1, 6)) \
             .filter(F.year("tpep_pickup_datetime") == 2025) \
             .filter(F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))

# 3. View denoising results
print("Denoised data row count:", df_clean.count())
print("Denoised data preview:")
df_clean.show(5)

# 4. Output processed dataset
output_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/cleaned_yellow_final_merged"
df_clean.write.mode("overwrite").parquet(output_path)

print("\n✅ Yellow taxi data denoising completed!")
print("✅ Cleaned dataset saved to:", output_path)

Spark Version: 4.1.0
Platform: Databricks
原始数据行数: 40840053
去噪后数据行数: 29472233
去噪后数据预览：
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|month|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----+
|       2| 2025-01-01 00:3

In [0]:
# Databricks Notebook
# Check Step1 Yellow Taxi Data Cleaning Results

from pyspark.sql import functions as F

check_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/cleaned_yellow_final_merged"
df_check = spark.read.parquet(check_path)

print("====== 🔍 Yellow Taxi Step1 Cleaning Check ======\n")

total = df_check.count()
print(f"Total data rows: {total}")

distinct_count = df_check.distinct().count()
print(f"Row count after deduplication: {distinct_count}")
print(f"Completely free of duplicates: {total == distinct_count}")

null_pickup = df_check.filter(F.col("tpep_pickup_datetime").isNull()).count()
print(f"Rows with NULL pickup time: {null_pickup}")

null_dropoff = df_check.filter(F.col("tpep_dropoff_datetime").isNull()).count()
print(f"Rows with NULL dropoff time: {null_dropoff}")

invalid_distance = df_check.filter(F.col("trip_distance") <= 0).count()
print(f"Rows with trip_distance <= 0: {invalid_distance}")

invalid_fare = df_check.filter((F.col("fare_amount") <= 0) | (F.col("fare_amount") > 1000)).count()
print(f"Rows with invalid fare amount: {invalid_fare}")

invalid_passenger = df_check.filter(~F.col("passenger_count").between(1,6)).count()
print(f"Rows with invalid passenger count: {invalid_passenger}")

invalid_year = df_check.filter(F.year("tpep_pickup_datetime") != 2025).count()
print(f"Rows not from 2025: {invalid_year}")

invalid_time_order = df_check.filter(F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime")).count()
print(f"Rows with incorrect time order: {invalid_time_order}")

print("\n====== ✅ Check Complete ======")

====== 🔍 黄色出租车 Step1 清洗结果检查 ======

总数据行数: 29472233
去重后行数: 29472233
是否完全无重复: True
上车时间为空行数: 0
下车时间为空行数: 0
行程距离<=0行数: 0
车费异常行数: 0
乘客数异常行数: 0
非2025年数据行数: 0
时间顺序错误行数: 0

====== ✅ 检查完成 ======


In [0]:
# Databricks Notebook
# 6009 Assignment 1 - Step 2 Time Feature Extraction (Yellow Taxi Time Dimension)

from pyspark.sql import functions as F

# ====================== Read cleaned yellow taxi dataset ======================
cleaned_data_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/cleaned_yellow_final_merged"
df_clean = spark.read.parquet(cleaned_data_path)

print("✅ Successfully read cleaned yellow taxi dataset")
print(f"Dataset row count: {df_clean.count()}")
df_clean.printSchema()
df_clean.show(3)

# ====================== Time Dimension Extraction ======================
# Extract four time features based on tpep_pickup_datetime (yellow taxi specific field)
df_with_time = df_clean \
    .withColumn("Hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("DayOfWeek", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("Month", F.month("tpep_pickup_datetime")) \
    .withColumn("IsWeekend", 
                F.when(F.dayofweek("tpep_pickup_datetime").isin(1, 7), True)
                 .otherwise(False))

print("\n✅ Time dimension extraction completed!")
print("New field preview:")
df_with_time.select("tpep_pickup_datetime", "Hour", "DayOfWeek", "Month", "IsWeekend").show(10)

# ====================== Save Processed Dataset ======================
output_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/yellow_tripdata_with_time_features"
df_with_time.write.mode("overwrite").parquet(output_path)

print("\n✅ Yellow taxi dataset with time features saved to:")
print(output_path)

# ====================== ✅ Statistics of final exported dataset: row count + column count + all column names ======================
# Read the final saved file
df_final = spark.read.parquet(output_path)

# Row count
row_count = df_final.count()

# Column count
col_count = len(df_final.columns)

# All column names
column_names = df_final.columns

print("\n" + "="*60)
print("📊 Final Exported Dataset Statistics (Yellow Taxi + Time Features)")
print("="*60)
print(f"✅ Total rows: {row_count}")
print(f"✅ Total columns：{col_count}")
print(f"✅ All column names：")
for idx, col in enumerate(column_names, 1):
    print(f"   {idx}. {col}")
print("="*60)

✅ 成功读取清洗后黄色出租车数据集
数据集行数: 29472233
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- month: long (nullable = true)

+-

In [0]:
# ====================== Check Yellow Taxi Time Features ======================
# 1. Count orders per hour (check if Hour is normal, not all zeros)
df_with_time.groupBy("Hour").count().orderBy("Hour").show(24)

# 2. Random preview of 20 records (check time, hour, day of week are correct)
df_with_time.select(
    "tpep_pickup_datetime", 
    "Hour", 
    "DayOfWeek", 
    "Month", 
    "IsWeekend"
).orderBy(F.rand()).show(20)

# 3. Count orders for weekend vs. weekday (optional, for assignment report)
df_with_time.groupBy("IsWeekend").count().show()

+----+-------+
|Hour|  count|
+----+-------+
|   0| 764766|
|   1| 496849|
|   2| 315444|
|   3| 203534|
|   4| 138737|
|   5| 169947|
|   6| 376781|
|   7| 755573|
|   8|1054097|
|   9|1245880|
|  10|1385000|
|  11|1497127|
|  12|1626115|
|  13|1698486|
|  14|1821305|
|  15|1879472|
|  16|1924145|
|  17|2075165|
|  18|2122727|
|  19|1870062|
|  20|1710254|
|  21|1718672|
|  22|1516295|
|  23|1105800|
+----+-------+

+--------------------+----+---------+-----+---------+
|tpep_pickup_datetime|Hour|DayOfWeek|Month|IsWeekend|
+--------------------+----+---------+-----+---------+
| 2025-11-10 19:48:26|  19|        2|   11|    false|
| 2025-09-24 18:52:09|  18|        4|    9|    false|
| 2025-03-06 11:02:15|  11|        5|    3|    false|
| 2025-11-14 00:13:48|   0|        6|   11|    false|
| 2025-11-11 12:47:42|  12|        3|   11|    false|
| 2025-07-18 20:08:38|  20|        6|    7|    false|
| 2025-10-29 07:44:37|   7|        4|   10|    false|
| 2025-05-07 21:28:25|  21|        4|  